# N087 · 二分图、增广路与匹配

**目标：** 用交替路径扩展匹配并识别适用图结构。

**先修：** N080, N081。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 二染色；奇环；匹配；交替/增广路；Kőnig定理的条件。

**配套讲解来源：** [同名逐章意见](../../comment/087_bipartite_graphs_matching.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章解决两个相关的问题。

**问题一：一张图是不是"二分图"？** 二分图就是能把所有点分成左右两堆，使得**每条边都恰好横跨两堆**（没有一堆内部自己连自己的边）。生活里的典型场景：一群学生和一组课程，"学生选课"关系里边只连学生和课程，学生之间、课程之间没有边。判断一张任意给定的图能不能这样分堆，就是 `is_bipartite` 干的事。

**问题二：二分图里最多能配成多少对？** 在左堆和右堆之间，我们要选出尽量多的边，但**每个点最多只能属于一条选中的边**——这叫"最大匹配"。典型场景：m 个工人、n 项任务，每个工人只会其中几项任务，每人最多干一项、每项最多一人干，问最多安排几项工作。

给一个具体到数字的小例子（就是 notebook “运行示例”部分 用的）：左边 3 个点，`left_adj=[[0,1],[0],[1,2]]`，意思是左点 0 会干右点 0 和 1，左点 1 只会右点 0，左点 2 会右点 1 和 2。右边有 3 个点（编号 0、1、2）。

- 直觉配法：左 0→右 0，左 1 就没活干了（它只会右 0）；左 2→右 1。配成 2 对，但右 2 闲置。
- 更优配法：左 0→右 1，左 1→右 0，左 2→右 2。三对全部配上，答案 **3**。
- 关键在于：第一个"直觉配法"把右 0 独占了，堵死了左 1；算法的本事就是发现"让左 0 挪去干右 1，把右 0 腾出来给左 1"，这一挪就多配出一对。这个"挪"的操作就叫增广，是本章的灵魂。

## 2. 基础知识：先读懂这些词

- **二分图（bipartite graph）**：能把顶点分成两堆、所有边都跨堆的图。判定它的等价说法：图能被"二染色"。
- **二染色**：给每个点涂红或蓝两种颜色之一，要求**每条边的两端颜色不同**。涂得成就是二分图，涂不成就不是。
- **奇环**：长度为奇数的环（比如三角形三 条边）。一个图存在奇环当且仅当它不是二分图——因为沿着奇环走一圈，颜色必须交替，绕一圈回来会要求起点既是红又是蓝，矛盾。
- **匹配（matching）**：一组被选中的边，要求任何两条选中边不共享端点（也就是每个点至多参加一条选中边）。
- **最大匹配**：所有匹配里边数最多的那个。
- **交替路径**：一条从某个左点出发的路，边依次是"未匹配边、匹配边、未匹配边、匹配边……"交替出现。想象你在配好对的关系网里走路：先走一条没配对边，再沿一条已配对边，如此往复。
- **增广路（augmenting path）**：一条交替路径，**起点是空闲的左点，终点是空闲的右点**（两端都没被匹配占用）。它的妙处在于：把这条路上所有边的"匹配/未匹配"状态全部翻转，匹配数恰好加一。
- **Berge 定理（“正确性与复杂度”部分 论证的核心）**：一个匹配是最大匹配，当且仅当图中不再存在增广路。这给了算法一个清晰的终止条件：不停增广，直到找不到增广路为止。
- **Kőnig 定理（提及的条件）**：在二分图中，"最小点覆盖数 = 最大匹配数"。点覆盖指挑一组点，使每条边至少有一端被挑中。它只对二分图成立，这也是识别"适用图结构"的知识点（本章只要求知道这个联系，不展开证明）。
- **前驱数组 prev（重建路径用）**：BFS 过程中记下"我是从哪个左点走到这个右点的"，回溯它就能还原整条增广路。用显式队列 + 前驱记录代替递归，增广链再长也不会栈溢出（“思路推导”部分 特意强调这一点）。
- **-1 哨兵**：`left`/`right` 数组里 -1 表示"这个点还没配上对"（“正确性与复杂度”部分 明说这是契约）。

## 3. 思路推导：从小例子开始

### 判定二分图（染色法 BFS）

- **Step 1**：每个点初始无色（-1）。图可能不连通，所以要逐个点检查：遇到没染色的点就从它出发开一轮 BFS。
- **Step 2**：起点染 0 号色入队。每弹出一个点 u，看它所有邻居 v。
- **Step 3**：若 v 已染色且和 u 同色——立即返回 False（这条边两端同色，违反约束）。
- **Step 4**：若 v 无色——染成与 u 相反的颜色（`1-color[u]`）并入队。
- **Step 5**：所有点都处理完且没撞色，返回 True。

**手算演示**：`adj=[[1,2],[0,2],[0,1]]`（三点两两相连，三角形）。

- 从点 0 出发：color[0]=0，入队。
- 弹 0，看邻居 1：无色，染 1。看邻居 2：无色，染 1。此时 color=[0,1,1]。
- 弹 1，看邻居 0：颜色 0 ≠ 1，没问题。看邻居 2：`color[2]==color[1]`（都是 1）——同色！返回 False。
- 道理很直白：三角形就是奇环，绕一圈必然撞色。这正是测试第一条 `assert not is_bipartite([[1,2],[0,2],[0,1]])`。

### 最大匹配（逐个左点 BFS 找增广路）

- **Step 1**：`left[u]` 记左点 u 配给了哪个右点，`right[v]` 记右点 v 被哪个左点占了，初始全是 -1（都空闲）。
- **Step 2**：对每个左点 start 依次尝试增广：从 start 开始 BFS。队列里放的是"左点"，`prev` 记录"到达右点 v 之前走的未匹配边来自哪个左点"。
- **Step 3**：从队列左点 u 出发，遍历它会干的每个右点 v：若 v 已在 prev 里就跳过（防止绕圈）；否则记 `prev[v]=u`。**若 v 空闲（right[v]==-1），增广路找到了，end=v，收工**。
- **Step 4**：若 v 被占用，就让占用它的左点 `nxt=right[v]` 入队——意思是"你能不能挪一挪，把 v 让出来？"。交替性由此保证：u→v 是未匹配边，v→nxt 是匹配边，接下去 nxt 的搜索又是未匹配边。
- **Step 5**：找到 end 后，沿 prev 回溯翻转：把沿途每条"未匹配边"变成匹配边。回溯中顺带把被顶替的旧左点链式上移（代码里 `end=old` 的技巧，下面细讲）。

**手算演示**（“运行示例”部分 的例子）：`left_adj=[[0,1],[0],[1,2]]`，右点 3 个。

- **处理左点 0**：BFS 从 0 出发。邻居右 0：空闲！end=0，prev[0]=左0。翻转：left[0]=右0，right[0]=左0。匹配数 1。当前配对：L0–R0。
- **处理左点 1**：BFS 从 1 出发。邻居右 0：被占（right[0]=左0），记 prev[0]=左1，把占用者左 0 入队。继续弹左 0：它的邻居右 0 已在 prev，跳过；邻居右 1：空闲！end=1，prev[1]=左0。开始翻转：这条增广路是 左1→右0 (未配) →左0→右1 (未配)。翻转后 left[1]=右0、left[0]=右1。匹配数 2。当前配对：L0–R1，L1–R0。**注意这一步就是"挪窝"：左 0 从右 0 挪到右 1，把右 0 腾给左 1。**
- **处理左点 2**：BFS 从 2 出发。邻居右 1：被左 0 占，记 prev[1]=左2，左 0 入队。弹左 0：邻居右 0 被左 1 占（prev[0]=左0，左1 入队）；邻居右 1 已在 prev，跳过。弹左 1：邻居右 0 已在 prev，跳过。队空了，end 还是 -1——找不到增广路？等等，重看：左 2 的邻居是 [1,2]，右 1 之后还会试右 2。回到起点弹左 2 时先看右 1（被占、入队左 0），再看右 2：**空闲！** end=2，prev[2]=左2。翻转：left[2]=右2。匹配数 3。最终 L0–R1，L1–R0，L2–R2。
- “运行示例”部分 的表格正是最终答案：左 0 → 右 1，左 1 → 右 0，左 2 → 右 2。

## 4. 核心代码：is_bipartite

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def is_bipartite(adj):
    color = [-1] * len(adj)
    for s in range(len(adj)):
        if color[s] != -1:
            continue
        color[s] = 0
        q = deque([s])
        while q:
            u = q.popleft()
            for v in adj[u]:
                if color[v] == color[u]:
                    return False
                if color[v] == -1:
                    color[v] = 1 - color[u]
                    q.append(v)
    return True
```

### 逐段读懂代码

### 函数 1：`is_bipartite(adj)`

```python
from collections import deque

def is_bipartite(adj):
    color = [-1] * len(adj)
    for s in range(len(adj)):
        if color[s] != -1:
            continue
        color[s] = 0
        q = deque([s])
```

- 先导入双端队列当 BFS 队列用（popleft 是 O(1)）。
- `color=[-1]*len(adj)`：-1 表示还没染色，0 和 1 是两种颜色。
- 外层循环逐点扫：`if color[s]!=-1: continue` 跳过已染色的点。这个循环保证**每个连通分量都会被处理**——图不连通时一次 BFS 只能覆盖一个分量。
- 起点 `color[s]=0`：第一个点染什么色无所谓（0 也行），因为两种颜色地位对称。

```python
while q:
            u=q.popleft()
            for v in adj[u]:
                if color[v]==color[u]: return False
                if color[v]==-1: color[v]=1-color[u]; q.append(v)
    return True
```

- 弹出 u，逐个看邻居 v。
- 第一条判断是**立刻判死刑**：v 已染色且和 u 撞色，说明这条边两端会同色，直接返回 False，不用再折腾。
- 第二条：v 没染色就染成相反色 `1-color[u]`（0 变 1、1 变 0）并入队。
- 队列跑空、外层跑完都没撞色，返回 True。

### 函数 2：`maximum_bipartite_matching(left_adj, right_size)`

```python
def maximum_bipartite_matching(left_adj, right_size):
    left = [-1] * len(left_adj)
    right = [-1] * right_size
```

- `left[u]`：左点 u 当前配到的右点编号；`right[v]`：右点 v 当前被哪个左点占用。-1 表示空闲（这是 “正确性与复杂度”部分 明说的契约）。

```python
for start in range(len(left)):
        q=deque([start]); seen={start}; prev={}; end=-1
```

- 每个左点轮流做一次"找增广路"的尝试。`q` 是左点队列；`seen` 是已入过队的左点集合（防绕圈）；`prev` 是字典：右点 v → 到 v 走的那条未匹配边的左端点；`end=-1` 表示还没找到空闲右点。

```python
while q and end==-1:
            u=q.popleft()
            for v in left_adj[u]:
                if v in prev: continue
                prev[v]=u
                if right[v]==-1: end=v; break
                nxt=right[v]
                if nxt not in seen: seen.add(nxt); q.append(nxt)
```

- 循环条件 `while q and end==-1`：队列空（没路了）或 end 找到（成功）都停。
- `if v in prev: continue`：这个右点已经被考察过，再走一次只会绕圈，跳过。prev 同时兼任"右点已访问"标记。
- `prev[v]=u`：记下"我是经 u 的未匹配边到达 v 的"，回溯全靠它。
- `if right[v]==-1: end=v; break`：v 空闲——增广路终点到手，立刻 break 不再探索。
- 否则 v 被占用，`nxt=right[v]` 是占用者；`if nxt not in seen` 保证每个左点只入队一次，入队 nxt 意味着"请它考虑挪窝"。

```python
while end!=-1:
            u=prev[end]; old=left[u]
            left[u]=end; right[end]=u; end=old
```

- 这是**回溯翻转**的精髓，用一个 while 把整条增广路一段段翻完。以手算第二步为例：end=右1，prev[右1]=左0。第一圈：u=左0，old=left[左0]=右0（旧配对）；置 left[左0]=右1、right[右1]=左0；然后 `end=old=右0`——注意！end 变成了被顶下来的旧右点，循环继续。
- 第二圈：end=右0（≠-1），u=prev[右0]=左1，old=left[左1]=-1（左 1 原本空闲）；置 left[左1]=右0、right[右0]=左1；`end=old=-1`，循环结束。
- 你看，这个写法不需要显式存整条路径再倒序翻转，而是"边回溯边翻转"，被顶替的配对自动顺着链上传，最后 end 变 -1 自然收尾。紧凑但每一行都有含义。
- 若本次 BFS 失败（end 保持 -1），这个 while 一次都不执行，匹配原样不动——放弃这个左点，它就是配不上了。

```python
return sum(v!=-1 for v in left),left
```

- 返回 (匹配数, 每个左点的配对表)。匹配数就是 left 里不等于 -1 的格子数。

## 5. 分段实现：按顺序运行

**本章接口：** `is_bipartite(adj)`、`maximum_bipartite_matching(left_adj, right_size)`

### 导入本章使用的库

In [1]:
from collections import deque

### is_bipartite

In [2]:
def is_bipartite(adj):
    color = [-1] * len(adj)
    for s in range(len(adj)):
        if color[s] != -1:
            continue
        color[s] = 0
        q = deque([s])
        while q:
            u = q.popleft()
            for v in adj[u]:
                if color[v] == color[u]:
                    return False
                if color[v] == -1:
                    color[v] = 1 - color[u]
                    q.append(v)
    return True

### maximum_bipartite_matching

In [3]:
def maximum_bipartite_matching(left_adj, right_size):
    left = [-1] * len(left_adj)
    right = [-1] * right_size
    for start in range(len(left)):
        q = deque([start])
        seen = {start}
        prev = {}
        end = -1
        while q and end == -1:
            u = q.popleft()
            for v in left_adj[u]:
                if v in prev:
                    continue
                prev[v] = u
                if right[v] == -1:
                    end = v
                    break
                nxt = right[v]
                if nxt not in seen:
                    seen.add(nxt)
                    q.append(nxt)
        while end != -1:
            u = prev[end]
            old = left[u]
            left[u] = end
            right[end] = u
            end = old
    return (sum((v != -1 for v in left)), left)

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [4]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

adj=[[0,1],[0],[1,2]]; size,match=maximum_bipartite_matching(adj,3)
show_table(['左点','候选右点','最终匹配'],[(u,a,match[u]) for u,a in enumerate(adj)])

左点,候选右点,最终匹配
0,"[0, 1]",1
1,[0],0
2,"[1, 2]",2


## 7. 正确性、适用条件与复杂度

交替路径翻转后，内部点仍各配一条边，两个端点从空闲变为已配。若不存在增广路，却有更大匹配，两匹配的对称差必含一个增广路径，矛盾。因此逐次增广至不能增广即为最大匹配。染色时同色边正好违反二分约束。

**代价：** 染色O(V+E)；本实现每个左点一次O(V+E)搜索，总O(L(V+E))，辅助O(V)。邻接表的右点编号为0..right_size−1；匹配数组−1表示未匹配。

### 展开理解

**为什么翻转增广路能让匹配数恰好加一？** 我们盯着一条增广路看：起点是空闲左点，终点是空闲右点，中间的边交替是"未匹配、匹配"。翻转之后：原来未匹配的边（比原来匹配的边**多一条**，因为路的两端都是未匹配边出头）变成匹配，原来匹配的变成不匹配。中间的每个点原本各占一条匹配边，翻转后还是各占一条（只是换了一条），谁也没多没少；而两端的空闲点各自获得一条匹配边。净效果：匹配边多了一条，且依然没有任何点同时挂两条边——还是合法匹配。

**为什么"找不到增广路"就说明已经最大？** 反过来想：假如当前匹配不是最大的，也就是说存在一个边数更多的匹配 M'。把当前匹配 M 和 M' 放在一起对比，只看那些"属于恰好一个匹配"的边，它们构成的子图里每个点最多挂 2 条边（M 一条、M' 一条），所以这个子图只能由互不相交的路径和环组成。又因为 M' 的边更多，里面必然存在一条路径是"以 M' 的边开头、以 M' 的边结尾、两端点在 M 中空闲"的——这恰好就是一条关于 M 的增广路。矛盾：我们刚才说找不到增广路。所以找不到增广路时，当前匹配必然已是最大。这就是"逐次增广直到不能增广"这个策略的正确性保证。

**染色判定为什么对？** 染色一旦撞色就返回 False，这条同色边实实在在违反了"每条边两端异色"的要求，False 报得有理。而全部染完没撞色时，颜色 0 和颜色 1 就是现成的两堆划分，True 也报得有理。加上"奇环必撞色"（沿环交替数一圈差半条边），这套判定和二分图的定义完全等价。

**复杂度，用具体数字说：**

- `is_bipartite` 是 O(V+E)：每个点入队一次、每条边被两端各看一次。比如 10⁵ 个点、2×10⁵ 条边，也就是几十万次基本操作，瞬间完成。
- 匹配的实现是每个左点跑一次 O(V+E) 的 BFS，共 L 个左点，总计 O(L×(V+E))。它不是最快的（更高级的 Hopcroft–Karp 是 O(E√V)），但胜在结构清晰。举例：L=V=100、E=1000 时约 100×1100≈10⁵ 次操作，毫无压力；若 L=V=10⁴、E=10⁵，则约 10⁹ 次，就会明显吃力——那种规模才需要换更快的算法。辅助空间 O(V)，就是那几个队列和字典。
- “正确性与复杂度”部分 还明确了输入契约：邻接表里的右点编号必须是 0 到 right_size-1，匹配数组用 -1 表示未匹配。

## 8. 单元测试：每个用例在检查什么

```python
assert not is_bipartite([[1, 2], [0, 2], [0, 1]])
```
反例（特殊结构）：三角形，含 3 条边的奇环。奇环不可二染色，必须返回 False。这条防止实现"只会返回 True"。

```python
assert is_bipartite([[1, 3], [0, 2], [1, 3], [0, 2]])
```
正例（正常值）：经典的四点"跑道"图（0-1-2-3-0，偶数环）。0、2 一色，1、3 另一色即可，应返回 True。

```python
assert maximum_bipartite_matching([[0, 1], [0]], 2)[0] == 2
```
最小"挪窝"场景：左 0 会右 0 和 1，左 1 只会右 0。如果算法先让左 0 占了右 0 就死心，只能配 1 对；正确做法是让左 0 挪去右 1，配成 2 对。这条专测增广路的翻转逻辑。

```python
from itertools import product

def brute(adj, m, u=0, used=0):
    if u == len(adj):
        return 0
    return max([brute(adj, m, u + 1, used)] + [1 + brute(adj, m, u + 1, used | 1 << v) for v in adj[u] if not used >> v & 1])
```
暴力参考解：递归枚举每个左点"不配或配给某个还没被占的右点"，用位掩码 used 记录右点占用情况（`used>>v&1` 检查 v 是否已用，`used|(1<<v)` 表示占用 v）。它指数级慢但绝对正确，用来给快速算法对答案。

```python
for bits in product(range(2), repeat=9):
    adj = [[v for v in range(3) if bits[u * 3 + v]] for u in range(3)]
    size, match = maximum_bipartite_matching(adj, 3)
    assert size == brute(adj, 3)
    assert len({v for v in match if v != -1}) == size
    assert all((v == -1 or v in adj[u] for u, v in enumerate(match)))
```
穷举所有 2⁹=512 张"3 左 × 3 右"的二分图（9 个可能的连接位各有 0/1 两种）。每张图检查三件事：一是结果大小和暴力解一致；二是 match 里非 -1 的右点互不重复（`len({...})==size`，即没有两个左点配同一个右点）；三是每个配对都是真实存在的边（v 在 adj[u] 里）。512 张图全覆盖，连"某些左点没有任何邻居"这种退化输入都包含在枚举里（全 0 位图）。

最后 `print('N087: 所有本章断言通过')` 供肉眼确认全绿。

In [5]:
assert not is_bipartite([[1, 2], [0, 2], [0, 1]])

assert is_bipartite([[1, 3], [0, 2], [1, 3], [0, 2]])

assert maximum_bipartite_matching([[0, 1], [0]], 2)[0] == 2

from itertools import product

def brute(adj, m, u=0, used=0):
    if u == len(adj):
        return 0
    return max([brute(adj, m, u + 1, used)] + [1 + brute(adj, m, u + 1, used | 1 << v) for v in adj[u] if not used >> v & 1])

for bits in product(range(2), repeat=9):
    adj = [[v for v in range(3) if bits[u * 3 + v]] for u in range(3)]
    size, match = maximum_bipartite_matching(adj, 3)
    assert size == brute(adj, 3)
    assert len({v for v in match if v != -1}) == size
    assert all((v == -1 or v in adj[u] for u, v in enumerate(match)))

print('N087: 所有本章断言通过')

N087: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 把考场冲突图按列奇偶划分。

**练习 2：** 比较最大匹配与贪心随便配对。

<details>
<summary>展开思路提示（不是完整答案）</summary>

**练习 1：把考场冲突图按列奇偶划分。**
思路方向：这是 LeetCode 1349 的场景——考场座位成网格，每个学生和其上下左右的学生不能同时考试，问最多多少人能同时考。提示：给每个座位按 (行,列) 染色，行号加列号是偶数的一堆、奇数的一堆，那么"上下左右相邻"的座位必然分属两堆——这正好是一个二分图，"不能同时考试"就是边。于是问题转化为在这张二分图上求**最大独立集**（挑尽量多的点互不相邻）；在二分图里由 Kőnig 定理推论，最大独立集 = 总点数 − 最大匹配。你可以拿一个 2×2 或 3×3 的小网格手算一遍：哪些座位互相冲突、匹配是谁、最后能坐几人。

**练习 2：比较最大匹配与贪心随便配对。**
思路方向：贪心策略是"按顺序给每个左点配上它第一个还空闲的邻居，配不上就跳过"。你要构造一个贪心会输的例子。提示：我们第一节的招牌例子 `[[0,1],[0]]` 就够了——贪心让左 0 占右 0，左 1 就废了，只得 1 对；最大匹配是 2 对。再手算展示本章算法如何通过增广路"让左 0 挪窝"扭转败局。进一步可以讨论：贪心在某些图上恰好也能得到最优（比如所有左点的候选集互不重叠时），说明贪心错在"局部选择堵死别人"而不总有错。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [6]:
# 完整实现：本单元格不依赖其他单元格。
from collections import deque

def is_bipartite(adj):
    color = [-1] * len(adj)
    for s in range(len(adj)):
        if color[s] != -1:
            continue
        color[s] = 0
        q = deque([s])
        while q:
            u = q.popleft()
            for v in adj[u]:
                if color[v] == color[u]:
                    return False
                if color[v] == -1:
                    color[v] = 1 - color[u]
                    q.append(v)
    return True

def maximum_bipartite_matching(left_adj, right_size):
    left = [-1] * len(left_adj)
    right = [-1] * right_size
    for start in range(len(left)):
        q = deque([start])
        seen = {start}
        prev = {}
        end = -1
        while q and end == -1:
            u = q.popleft()
            for v in left_adj[u]:
                if v in prev:
                    continue
                prev[v] = u
                if right[v] == -1:
                    end = v
                    break
                nxt = right[v]
                if nxt not in seen:
                    seen.add(nxt)
                    q.append(nxt)
        while end != -1:
            u = prev[end]
            old = left[u]
            left[u] = end
            right[end] = u
            end = old
    return (sum((v != -1 for v in left)), left)

# 示例与回归测试
assert not is_bipartite([[1, 2], [0, 2], [0, 1]])

assert is_bipartite([[1, 3], [0, 2], [1, 3], [0, 2]])

assert maximum_bipartite_matching([[0, 1], [0]], 2)[0] == 2

from itertools import product

def brute(adj, m, u=0, used=0):
    if u == len(adj):
        return 0
    return max([brute(adj, m, u + 1, used)] + [1 + brute(adj, m, u + 1, used | 1 << v) for v in adj[u] if not used >> v & 1])

for bits in product(range(2), repeat=9):
    adj = [[v for v in range(3) if bits[u * 3 + v]] for u in range(3)]
    size, match = maximum_bipartite_matching(adj, 3)
    assert size == brute(adj, 3)
    assert len({v for v in match if v != -1}) == size
    assert all((v == -1 or v in adj[u] for u, v in enumerate(match)))

print('N087: 所有本章断言通过')

N087: 所有本章断言通过


## 11. 代表题与迁移

- **785. Is Graph Bipartite?（判断二分图）**：这题直接练本章 `is_bipartite` 的场景——给邻接表判断能否二染色，注意要用外层循环覆盖不连通的图。
- **886. Possible Bipartition（可能的二分法）**：这题练的是"先把 dislike 关系建成图，再套二染色判定"的两步套路，和 785 同源，考点在建模：人分两组、互不喜欢的人必须异组。
- **1349. Maximum Students Taking Exam（参加考试的最大学生数，extension）**：这题是本章知识的延伸综合——把座位网格按行列奇偶建成二分图后，用"最大独立集 = 点数 − 最大匹配"（Kőnig 定理的推论）求解，正好对应练习 1 的思路。notebook 标注为 extension，说明需要额外一步转化，不是直接套接口。

**题面入口：** [1349](https://leetcode.com/problems/maximum-students-taking-exam/)

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。